# OmniGen v1 — Multi-Image VTO Server

This notebook hosts OmniGen v1 as a Gradio API. It accepts **2-3 person images + 1 garment image** for better identity preservation.

**GPU:** T4 works, A100 is faster. Go to Runtime → Change runtime type → GPU.

**How to use:**
1. Run all cells
2. Copy the `https://xxxxx.gradio.live` URL
3. Set it as `OMNIGEN_COLAB_URL` secret in Supabase

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q gradio Pillow requests

# Clone OmniGen
import os
if not os.path.exists('OmniGen'):
    !git clone https://github.com/VectorSpaceLab/OmniGen.git
os.chdir('OmniGen')
!pip install -q -e .

print('Dependencies installed!')

In [ ]:
#@title 2. Load OmniGen model
import torch
from OmniGen import OmniGenPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'VRAM: {vram:.1f} GB')

# Use quantized model for T4, full model for A100
if vram < 20:
    print('Low VRAM detected, using 4-bit quantized model...')
    model_name = 'Shitao/OmniGen-v1'
    use_kv_cache = False
    offload = True
else:
    model_name = 'Shitao/OmniGen-v1'
    use_kv_cache = True
    offload = False

pipe = OmniGenPipeline.from_pretrained(model_name)
print(f'OmniGen v1 loaded from {model_name}!')

In [ ]:
#@title 3. Define generation function
from PIL import Image
import io
import base64
import requests
import tempfile
import json

def download_image(url, name='img'):
    """Download image from URL, save to temp file, return path."""
    response = requests.get(url, timeout=30)
    img = Image.open(io.BytesIO(response.content)).convert('RGB')
    # Resize to reasonable size to save VRAM
    max_dim = 1024
    if max(img.size) > max_dim:
        ratio = max_dim / max(img.size)
        img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)
    path = tempfile.mktemp(suffix='.jpg')
    img.save(path, 'JPEG', quality=90)
    return path

def generate_tryon(selfie_url, fullbody_url, garment_url, description='clothing item', steps=50):
    """Run OmniGen virtual try-on with multiple person references."""
    try:
        # Download all images
        input_images = []
        prompt_parts = []
        
        img_idx = 1
        
        # Selfie (face reference)
        if selfie_url and selfie_url.strip():
            selfie_path = download_image(selfie_url, 'selfie')
            input_images.append(selfie_path)
            prompt_parts.append(f'<img><|image_{img_idx}|></img>')
            img_idx += 1
        
        # Full body (body reference)
        fullbody_path = download_image(fullbody_url, 'fullbody')
        input_images.append(fullbody_path)
        prompt_parts.append(f'<img><|image_{img_idx}|></img>')
        body_idx = img_idx
        img_idx += 1
        
        # Garment
        garment_path = download_image(garment_url, 'garment')
        input_images.append(garment_path)
        garment_idx = img_idx
        
        # Build prompt
        if len(prompt_parts) == 2:  # selfie + fullbody
            prompt = (
                f'Generate a photorealistic image of the person shown in {prompt_parts[0]} '
                f'and {prompt_parts[1]} '
                f'wearing the {description} shown in <img><|image_{garment_idx}|></img>. '
                f'The person should be in a natural full-body standing pose, '
                f'preserving their exact face, skin tone, hair, and body proportions. '
                f'The clothing should fit naturally. Studio lighting, clean background, fashion photography.'
            )
        else:  # only fullbody
            prompt = (
                f'Generate a photorealistic image of the person shown in {prompt_parts[0]} '
                f'wearing the {description} shown in <img><|image_{garment_idx}|></img>. '
                f'Preserve their exact face, skin tone, hair, and body proportions. '
                f'Natural standing pose, studio lighting, fashion photography.'
            )
        
        print(f'Prompt: {prompt}')
        print(f'Input images: {len(input_images)}')
        
        # Run generation
        with torch.no_grad():
            output = pipe(
                prompt=prompt,
                input_images=input_images,
                num_inference_steps=steps,
                guidance_scale=3.0,
                img_guidance_scale=1.6,
                height=1024,
                width=768,
                seed=42,
                use_kv_cache=use_kv_cache,
                offload_kv_cache=True,
                offload_model=offload,
            )
        
        result_img = output[0]
        
        # Convert to base64
        buf = io.BytesIO()
        result_img.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        
        return json.dumps({"success": True, "image_base64": b64})
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return json.dumps({"success": False, "error": str(e)})

print('Generation function ready!')

In [ ]:
#@title 4. Launch Gradio API Server
import gradio as gr

demo = gr.Interface(
    fn=generate_tryon,
    inputs=[
        gr.Textbox(label='Selfie URL (optional)', placeholder='https://...signed-url for face close-up...'),
        gr.Textbox(label='Full Body URL (required)', placeholder='https://...signed-url for full body...'),
        gr.Textbox(label='Garment URL (required)', placeholder='https://...signed-url for garment...'),
        gr.Textbox(label='Description', value='clothing item'),
        gr.Slider(minimum=20, maximum=100, value=50, step=5, label='Inference Steps'),
    ],
    outputs=gr.Textbox(label='Result JSON'),
    title='OmniGen v1 VTO API Server',
    description='Multi-image Virtual Try-On. Send selfie + full body + garment URLs.',
    api_name='tryon',
)

print('\n' + '='*60)
print('STARTING OMNIGEN SERVER...')
print('Copy the public URL below and set it as')
print('OMNIGEN_COLAB_URL in your Supabase secrets.')
print('='*60 + '\n')

demo.launch(share=True, quiet=False)